# Autoencoder Example 3 — Convolutional Autoencoder

The basic autoencoder flattened each image into a 784-vector and used dense layers — which
throws away the 2D spatial structure. A **convolutional autoencoder** keeps the image as a
2D grid and uses convolutions, so it respects locality and produces noticeably **sharper**
reconstructions with far fewer parameters.

## Architecture

- **Encoder**: `Conv2D` layers with `strides=2` shrink the image `28×28 → 14×14 → 7×7`
  while increasing channel depth — the spatial bottleneck.
- **Decoder**: `Conv2DTranspose` layers with `strides=2` grow it back `7×7 → 14×14 →
  28×28`, ending in a `sigmoid` so pixels land in `[0, 1]`.

(If this up/down-sampling pattern looks familiar, it's the same machinery the DCGAN's
generator and discriminator use.)

> Tested with `tensorflow==2.1`. Trains in under a minute on CPU.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

tf.random.set_seed(0)
print('TensorFlow', tf.__version__)

## 1. Data

Convolutions expect a spatial tensor, so we keep the `28×28×1` shape (no flattening).

In [ ]:
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32').reshape(-1, 28, 28, 1) / 255.0
x_test = x_test.astype('float32').reshape(-1, 28, 28, 1) / 255.0
print('Train:', x_train.shape)

## 2. Model

In [ ]:
autoencoder = tf.keras.Sequential([
    layers.Input((28, 28, 1)),
    # Encoder
    layers.Conv2D(32, 3, strides=2, padding='same', activation='relu'),  # 28 -> 14
    layers.Conv2D(64, 3, strides=2, padding='same', activation='relu'),  # 14 -> 7
    # Decoder
    layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu'),  # 7 -> 14
    layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu'),  # 14 -> 28
    layers.Conv2D(1, 3, padding='same', activation='sigmoid'),
])
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
autoencoder.summary()

## 3. Train

In [ ]:
autoencoder.fit(x_train, x_train,
                epochs=15, batch_size=256, shuffle=True,
                validation_data=(x_test, x_test), verbose=2)

## 4. Original vs reconstruction

Compare with the dense autoencoder's output — the convolutional version's reconstructions
should be visibly crisper.

In [ ]:
n = 10
recon = autoencoder.predict(x_test[:n])
plt.figure(figsize=(n * 1.5, 3))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray'); plt.axis('off')
    if i == 0: ax.set_title('original', loc='left')
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(recon[i].reshape(28, 28), cmap='gray'); plt.axis('off')
    if i == 0: ax.set_title('reconstructed', loc='left')
plt.show()

### Things to try

- Add a third down/up-sampling stage for a tighter bottleneck.
- Combine with example 2's noise to build a **convolutional denoising** autoencoder.
- Insert a `Flatten` + `Dense` bottleneck to get an explicit latent vector you can sample.
- Train on a richer dataset (Fashion-MNIST) to see the sharpness gain more clearly.